In [134]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from google.colab import files
import io
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

In [135]:
uploaded = files.upload()
print(uploaded)
# anaemia_outputs.csv #

file_name = list(uploaded.keys())[0]
print(f'Uploaded file name: {file_name}')

Saving anaemia_outputs.csv to anaemia_outputs (9).csv
{'anaemia_outputs (9).csv': b'Number,Sex,%Red Pixel,%Green pixel,%Blue pixel,Hb,Anaemic\r\n1,M,43.2555,30.8421,25.9025,6.3,Yes\r\n2,F,45.6033,28.19,26.2067,13.5,No\r\n3,F ,45.0107,28.9677,26.0215,11.7,No\r\n4,F,44.5398,28.9899,26.4703,13.5,No\r\n5,M ,43.287,30.6972,26.0158,12.4,No\r\n6,M,45.0994,27.9645,26.9361,16.2,No\r\n7,F,43.1457,30.1628,26.6915,8.6,Yes\r\n8,F ,43.6103,29.1099,27.2798,10.3,No\r\n9,F,45.0423,29.166,25.7918,13,No\r\n10,F,46.5143,27.4282,26.0575,9.7,Yes\r\n11,F,45.3506,29.1248,25.5246,12.6,No\r\n12,F,44.4062,28.9298,26.664,15.4,No\r\n13,F,44.9642,30.5279,24.5079,4.8,Yes\r\n14,M ,45.0484,31.1049,23.8467,9,Yes\r\n15,M ,46.9942,26.0496,26.9562,14.6,No\r\n16,M,45.5842,28.7311,25.6848,14,No\r\n17,F,42.5358,30.1604,27.3039,10,Yes\r\n18,F,44.0957,29.9973,25.907,8.3,Yes\r\n19,F,45.7104,27.5693,26.7204,13.6,No\r\n20,F ,40.9365,31.9687,27.0948,9.9,Yes\r\n21,F,44.9116,30.3761,24.7123,11.6,No\r\n22,F ,43.4225,29.7889,26.7886,1

In [136]:
df = pd.read_csv(io.BytesIO(uploaded[file_name]))
print(df.columns)
df.head()

Index(['Number', 'Sex', '%Red Pixel', '%Green pixel', '%Blue pixel', 'Hb',
       'Anaemic'],
      dtype='object')


,Number,Sex,%Red Pixel,%Green pixel,%Blue pixel,Hb,Anaemic
0,1,M,43.2555,30.8421,25.9025,6.3,Yes
1,2,F,45.6033,28.1900,26.2067,13.5,No
2,3,F,45.0107,28.9677,26.0215,11.7,No
3,4,F,44.5398,28.9899,26.4703,13.5,No
4,5,M,43.2870,30.6972,26.0158,12.4,No


In [137]:
# Since column 1 is listed as 'd_outputs'
df = pd.read_csv('anaemia_outputs.csv')

print(df.columns)
df.head()

Index(['Number', 'Sex', '%Red Pixel', '%Green pixel', '%Blue pixel', 'Hb',
       'Anaemic'],
      dtype='object')


,Number,Sex,%Red Pixel,%Green pixel,%Blue pixel,Hb,Anaemic
0,1,M,43.2555,30.8421,25.9025,6.3,Yes
1,2,F,45.6033,28.1900,26.2067,13.5,No
2,3,F,45.0107,28.9677,26.0215,11.7,No
3,4,F,44.5398,28.9899,26.4703,13.5,No
4,5,M,43.2870,30.6972,26.0158,12.4,No


In [138]:
if 'Number' in df.columns:
    print("Dropped 'Number' column")
    df = df.drop(columns=['Number'])
else:
    print("'nothing to drop")

if 'Sex' in df.columns:
    print("Dropped 'Sex' column")
    df = df.drop(columns=['Sex'])
else:
    print("nothing to drop")

df.head()

Dropped 'Number' column
Dropped 'Sex' column


,%Red Pixel,%Green pixel,%Blue pixel,Hb,Anaemic
0,43.2555,30.8421,25.9025,6.3,Yes
1,45.6033,28.1900,26.2067,13.5,No
2,45.0107,28.9677,26.0215,11.7,No
3,44.5398,28.9899,26.4703,13.5,No
4,43.2870,30.6972,26.0158,12.4,No


In [139]:
# Split the data into features and labels
X = df[['%Red Pixel', '%Green pixel', '%Blue pixel', 'Hb']]

# Convert Yes/No labels to 1/0
y = df['Anaemic'].map({'Yes': 1, 'No': 0})

print(X.head())
print(y.head())

   %Red Pixel  %Green pixel  %Blue pixel    Hb
0     43.2555       30.8421      25.9025   6.3
1     45.6033       28.1900      26.2067  13.5
2     45.0107       28.9677      26.0215  11.7
3     44.5398       28.9899      26.4703  13.5
4     43.2870       30.6972      26.0158  12.4
0    1
1    0
2    0
3    0
4    0
Name: Anaemic, dtype: int64


In [140]:
# Normalize/scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [141]:
# Custom Dataset class
class AnaemiaDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [142]:
# Create Dataset instance
dataset = AnaemiaDataset(X_scaled, y)

# Calculate sizes for each subset
train_size = int(0.6 * len(dataset))
valid_size = int(0.2 * len(dataset))
test_size = len(dataset) - train_size - valid_size

In [143]:
# Split the dataset
train_dataset, valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, valid_size, test_size])

# Create DataLoader instances
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [144]:
print(f'Train dataset size: {len(train_dataset)}')
print(f'Validation dataset size: {len(valid_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Train dataset size: 62
Validation dataset size: 20
Test dataset size: 22


In [145]:
# Define the neural network model
class AnaemiaModel(nn.Module):
    def __init__(self, input_size):
        super(AnaemiaModel, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

In [146]:
# Initialize the model, loss function, and optimizer
input_size = X_scaled.shape[1]
model = AnaemiaModel(input_size)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=.3)

In [147]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(all_labels, all_predictions)

print(f'Accuracy: {accuracy * 100:.2f}%')

Accuracy: 95.45%


In [148]:
# Training the model with validation
num_epochs = 100
model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    for features, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs.squeeze(), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation step
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for features, labels in valid_loader:
            outputs = model(features)
            loss = criterion(outputs.squeeze(), labels)
            val_loss += loss.item()

    print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {running_loss/len(train_loader)}, Validation Loss: {val_loss/len(valid_loader)}')
    model.train()

Epoch 1/100, Training Loss: 3.4324948489665985, Validation Loss: 1.0284489393234253
Epoch 2/100, Training Loss: 1.1631430387496948, Validation Loss: 1.4149205684661865
Epoch 3/100, Training Loss: 1.7539288848638535, Validation Loss: 3.8966033458709717
Epoch 4/100, Training Loss: 1.837084099650383, Validation Loss: 5.798227310180664
Epoch 5/100, Training Loss: 2.0056405905634165, Validation Loss: 10.000131607055664
Epoch 6/100, Training Loss: 1.5934307929128408, Validation Loss: 5.000002861022949
Epoch 7/100, Training Loss: 2.6786753833293915, Validation Loss: 5.000007152557373
Epoch 8/100, Training Loss: 3.4059409224428236, Validation Loss: 5.230648040771484
Epoch 9/100, Training Loss: 3.1251382734117215, Validation Loss: 10.0
Epoch 10/100, Training Loss: 3.333761936693918, Validation Loss: 10.0
Epoch 11/100, Training Loss: 3.28347384929657, Validation Loss: 10.0
Epoch 12/100, Training Loss: 3.3398664481937885, Validation Loss: 10.0
Epoch 13/100, Training Loss: 3.229166626930237, Valid

In [149]:
# Evaluating the model on the test set with additional metrics
model.eval()
all_labels = []
all_predictions = []

with torch.no_grad():
    for features, labels in test_loader:
        outputs = model(features)
        predicted = (outputs.squeeze() > 0.5).float()
        all_labels.extend(labels.numpy())
        all_predictions.extend(predicted.numpy())
        #total += labels.size(0)
        #correct += (predicted == labels).sum().item()

#accuracy = 100 * correct / total


conf_matrix = confusion_matrix(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)

print(f'Accuracy: {f1*100} % \n')
print(f'Confusion Matrix:\n{conf_matrix} \n')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')

Accuracy: 93.33333333333333 % 

Confusion Matrix:
[[14  1]
 [ 0  7]] 

Precision: 0.875
Recall: 1.0
F1 Score: 0.9333333333333333
